# RUN ALL — one notebook, start to finish

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yazanjer/An_Explainable_AI_Education/blob/main/notebooks/RUN_ALL.ipynb)

Runs the entire revised analysis in order: mount Drive, clone, install, test,
fetch the PISA data, smoke test, then the full budget.

**Run the cells top to bottom.** Each one prints what it did and stops with a
clear message if something is missing. Nothing here needs editing except the
two switches in Cell 1.

| Cell | Does | Time |
|---|---|---|
| 1 | Mount Drive, clone, install | 5 min |
| 2 | Test suite (no data needed) | 1 min |
| 3 | Download PISA 2018 | 10-20 min, once |
| 4 | Verify setup | 30 s |
| 5 | Smoke test (quick mode) | ~15 min |
| 6 | Full run | 20-40 h, resumable |
| 7 | Collect results | 1 min |

> **Runtime:** use **T4 GPU + High-RAM** (Runtime -> Change runtime type).
> Cell 3 reads a 1.8 GB file and will die on a standard runtime.

> **Disk:** budget **~3 GB** of Drive (500 MB zip + 1.8 GB .sav + 300 MB cache).


## Cell 1 — Mount, clone, install

In [ ]:
# The two switches you may want to change:
DRIVE_FOLDER = "An_Explainable_AI_Education"   # folder under MyDrive
RUN_FULL     = False                            # True = full budget in Cell 6

import os, subprocess, sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PROJECT  = Path("/content/drive/MyDrive") / DRIVE_FOLDER
REPO_URL = "https://github.com/yazanjer/An_Explainable_AI_Education.git"


def run(cmd, cwd=None):
    """Run a command and surface stderr, instead of a bare exit code."""
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print("command failed:", " ".join(cmd))
        print((r.stderr or r.stdout).strip()[:1500])
    return r


if (PROJECT / "src" / "vlpso_xai").exists():
    print(f"Repository already present at {PROJECT} - pulling any updates")
    run(["git", "-C", str(PROJECT), "pull", "--ff-only"])

else:
    PROJECT.mkdir(parents=True, exist_ok=True)
    non_empty = any(PROJECT.iterdir())

    if not non_empty:
        print(f"Cloning into {PROJECT} (a few minutes: Drive writes each file "
              "over the network)")
        r = run(["git", "clone", REPO_URL, str(PROJECT)])
    else:
        # git clone refuses a non-empty target. This happens whenever the Drive
        # folder was created by hand first (e.g. with data/raw already in it),
        # which is exactly what the README asks you to do. Initialise in place
        # and fetch instead -- same result, keeps any data already there.
        print(f"{PROJECT} exists and is not empty "
              f"({len(list(PROJECT.iterdir()))} entries) - fetching in place")
        run(["git", "init", "-q"], cwd=str(PROJECT))
        run(["git", "remote", "remove", "origin"], cwd=str(PROJECT))
        run(["git", "remote", "add", "origin", REPO_URL], cwd=str(PROJECT))
        r = run(["git", "fetch", "--depth", "1", "origin", "main"], cwd=str(PROJECT))
        if r.returncode == 0:
            r = run(["git", "checkout", "-f", "-B", "main", "origin/main"],
                    cwd=str(PROJECT))

    if not (PROJECT / "src" / "vlpso_xai").exists():
        raise SystemExit(
            "Could not obtain the repository.\n\n"
            "Most likely causes:\n"
            "  1. The repository is PRIVATE. Make it public, or clone with a\n"
            "     token:  https://<TOKEN>@github.com/yazanjer/An_Explainable_AI_Education.git\n"
            "  2. No network access from this runtime.\n"
            "  3. The push has not completed yet - check the repo in a browser.\n"
            "See the error text printed above."
        )

os.environ["VLPSO_PROJECT_ROOT"] = str(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
os.chdir(PROJECT)

print("\nInstalling core dependencies ...")
core = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                       str(PROJECT / "requirements-core.txt")],
                      capture_output=True, text=True)
if core.returncode != 0:
    print(core.stdout[-2000:], core.stderr[-2000:])
    raise SystemExit(
        "Core install FAILED - see the error above. Nothing will work until "
        "this is fixed. A single unsatisfiable version constraint makes pip "
        "abort the entire file, so check the requirement it names."
    )

print("Installing optional dependencies (failures here are non-fatal) ...")
opt = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                      str(PROJECT / "requirements-optional.txt")],
                     capture_output=True, text=True)
if opt.returncode != 0:
    print("  some optional packages did not install; the analysis will run "
          "with documented fallbacks:")
    for line in (opt.stderr or "").splitlines():
        if line.startswith("ERROR"):
            print("   ", line[:160])

# Repair a stale NumPy 1.x. pyreadstat, scipy and sklearn wheels on Colab are
# compiled against the NumPy 2 ABI; on 1.x they raise
# "numpy.dtype size changed ... Expected 96 ... got 88".
try:
    import numpy as _np
    _major = int(_np.__version__.split(".")[0])
except Exception:
    _major = 0
if _major < 2:
    print(f"Found numpy {_np.__version__} - upgrading to 2.x ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--force-reinstall", "--no-cache-dir", "numpy>=2.0"],
                   check=False)
    raise SystemExit(
        "\n==> numpy was upgraded. RESTART THE RUNTIME NOW\n"
        "    (Runtime -> Restart session), then re-run this cell."
    )

try:
    import pyreadstat  # noqa: F401
    print("numpy", _np.__version__, "and pyreadstat import cleanly")
except ImportError as exc:
    raise SystemExit(
        f"{exc}\n\n==> If the core install above reported no error, RESTART "
        "THE RUNTIME and re-run this cell."
    )

## Cell 2 — Test suite

Runs on synthetic data, so it needs no PISA file. **146 tests should pass.**
If they do, the environment is sound and the leakage guards are working.

This includes `test_leakage_detection_power.py`, which is calibrated against a
deliberately-leaky reference implementation — so it can actually detect
leakage rather than merely asserting its absence.

In [ ]:
!python -m pytest tests/ -q --tb=short

print("\nIf you see '146 passed', the environment is verified.")

## Cell 3 — Get the PISA 2018 data

Downloads from the OECD and caches to Drive. **Runs once**; later runs reuse
the cache. The data is not redistributed with the repository — the OECD licence
permits download from their site, not republication.

In [ ]:
from pathlib import Path
import subprocess

RAW = cfg.paths.data_raw
RAW.mkdir(parents=True, exist_ok=True)
found = list(RAW.rglob("CY07_MSU_STU_QQQ.sav"))

if found:
    print(f"Already present: {found[0]} ({found[0].stat().st_size/1e9:.1f} GB)")
else:
    zp = RAW / "SPSS_STU_QQQ.zip"
    if not zp.exists():
        print("Downloading ~500 MB from the OECD ...")
        subprocess.run(["curl", "-L", "-o", str(zp),
                        cfg.section("data", "source", "student_zip_url")], check=True)
    print("Extracting (~1.8 GB) ...")
    subprocess.run(["unzip", "-o", "-q", str(zp), "-d", str(RAW)], check=True)
    found = list(RAW.rglob("CY07_MSU_STU_QQQ.sav"))
    if found:
        zp.unlink()          # reclaim 500 MB of Drive
        print("Removed the zip to save space.")

assert found, (
    "CY07_MSU_STU_QQQ.sav not found. Download it manually from\n"
    "https://www.oecd.org/en/data/datasets/pisa-2018-database.html\n"
    f"and place it under {RAW}"
)
print(f"\nData ready: {found[0]}")

## Cell 4 — Verify before committing to a long run

Confirms the data is readable and the analytic sample is the size it should be
(35,943 Spanish students in 1,089 schools, before the missingness exclusion).

In [ ]:
import pyreadstat
from pathlib import Path

sav = list(cfg.paths.data_raw.rglob("CY07_MSU_STU_QQQ.sav"))[0]
_, meta = pyreadstat.read_sav(str(sav), metadataonly=True)
print(f"rows    : {meta.number_rows:,}")
print(f"columns : {len(meta.column_names):,}")

need = ["PV1MATH", "W_FSTUWT", "W_FSTURWT80", "CNTSCHID", "ST013Q01TA"]
absent = [c for c in need if c not in set(meta.column_names)]
assert not absent, f"expected columns missing: {absent}"
print(f"key columns present: {need}")
print("\nCell 4 OK - ready to run.")

## Cell 5 — Smoke test (quick mode)

**Never skip this.** Reduced budget: 1 plausible value, 3 folds, small grids.
If this completes, the full run will too.

Expected honest results — these are *supposed* to be far below the AUC of
1.0000 in the submitted manuscript, which was an artefact of outcome leakage:

| Task | AUC |
|---|---|
| Low vs. High | ~0.88 |
| Low vs. Medium | ~0.74 |
| Medium vs. High | ~0.69 |

In [ ]:
!python scripts/run_all.py --config quick

print("\n" + "=" * 70)
import pandas as pd
from pathlib import Path
flow = Path(cfg.paths.results) / "sample" / "sample_flow.csv"
if flow.exists():
    display(pd.read_csv(flow))

## Cell 6 — Full run

**20–40 T4-hours.** Colab will disconnect before this finishes, and that is
fine: every fold is checkpointed to `results/checkpoints/`, keyed by a hash of
the full configuration. Re-run this cell after a disconnect and it resumes,
skipping completed folds.

Set `RUN_FULL = True` in Cell 1 to enable it.

In [ ]:
# Streams output live so you can watch progress; safe to re-run after a
# disconnect (completed folds are skipped).
import subprocess, sys

if not RUN_FULL:
    print("RUN_FULL is False - skipping.")
    print("Set RUN_FULL = True in Cell 1 and re-run this cell when ready.")
else:
    proc = subprocess.Popen(
        [sys.executable, "scripts/run_all.py", "--config", "default"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print(f"\nexit code: {proc.returncode}")
    if proc.returncode != 0:
        print("Non-zero exit. Re-run this cell: checkpointing resumes where it stopped.")

## Cell 7 — Collect results

Lists every artefact with its SHA256 from `results/manifest.json`, so each
manuscript number is traceable to the code and config that produced it.

In [ ]:
import json, pandas as pd
from pathlib import Path

man = Path(cfg.paths.results) / "manifest.json"
if man.exists():
    m = json.loads(man.read_text())
    print(f"artefacts   : {m['n_artefacts']}")
    print(f"git commit  : {m['git_commit']}")
    print(f"config hash : {m['config_hash'][:12]}")
    display(pd.DataFrame(m["artefacts"])[["path", "sha256", "bytes"]].head(30))
else:
    print("No manifest yet - run Cell 5 or 6 first.")

print("\nTables for the manuscript are in results/tables/ as .csv and .tex")

---

## What to do with the output

`results/tables/` holds every manuscript table as both `.csv` and `.tex`.
Nothing is typed by hand.

**Read `CHANGELOG_REVISION.md` before writing anything up.** It records each
change, the reviewer comment it answers, and the numerical consequence —
including results that do not favour the proposed method. It also lists the
outstanding findings from an independent audit of this rebuild.

**Do not send `RESPONSE_TO_EDITOR.md` until the full run has finished**: it
cites budgets (5x5 folds, 10 seeds, 100 permutations) that only `--config
default` delivers.